In [2]:
import pandas as pd
import re

# -------------------------- 1. 配置参数：领域-关键词映射（可根据需求增删） --------------------------
FIELD_KEYWORDS = {
    "Python基础入门": ["入门", "从入门到实践", "从入门到精通", "笨办法", "你好!Python", "完全自学教程"],
    "机器学习与深度学习": ["机器学习", "深度学习", "神经网络", "scikit-learn", "AI+", "自然语言处理", "强化学习"],
    "数据分析与可视化": ["数据分析", "数据科学", "数据可视化", "Pandas", "NumPy", "Matplotlib", "数亦有道"],
    "办公自动化": ["办公自动化", "Python+Office", "ChatGPT办公", "玩转Excel", "高效办公"],
    "量化交易/金融分析": ["量化交易", "金融量化", "区块链量化", "vn.py", "交易策略", "量化投资"],
    "网络爬虫": ["网络爬虫", "爬虫", "Urllib", "BeautifulSoup", "Scrapy"],
    "青少年/趣味编程": ["小学生", "青少年", "趣味编程", "看漫画学Python", "创意编程"],
    "高阶开发与配套基础": ["全栈开发", "Web编程", "现代Python编程", "矩阵", "线性代数", "程序是怎样跑起来", "智能优化算法"]
}

# -------------------------- 2. 加载CSV数据 --------------------------
csv_file = "dangdang_python_books_clean.csv"
try:
    # 优先尝试utf-8编码，失败则用gbk（中文CSV常见编码）
    df = pd.read_csv(csv_file, encoding="utf-8")
except UnicodeDecodeError:
    df = pd.read_csv(csv_file, encoding="gbk")

# 填充空值（避免匹配时报错）
df["书名"] = df["书名"].fillna("")
df["简介"] = df["简介"].fillna("")
# 合并书名+简介为匹配文本
df["match_text"] = df["书名"] + " " + df["简介"]
# 统一为小写（取消大小写影响）
df["match_text"] = df["match_text"].str.lower()

# -------------------------- 3. 定义分类匹配函数 --------------------------
def classify_book(text):
    """根据文本匹配返回所属分类，无匹配则返回「其他/跨领域融合」"""
    text = text.lower()
    for field, keywords in FIELD_KEYWORDS.items():
        for kw in keywords:
            if re.search(kw.lower(), text):
                return field
    return "其他/跨领域融合"

# 为每本书添加分类标签
df["所属分类"] = df["match_text"].apply(classify_book)

# -------------------------- 4. 归总分类信息 --------------------------
classify_summary = df.groupby("所属分类").agg(
    图书数量=("书名", "count"),
    图书列表=("书名", list),
    代表作者=("作者", lambda x: list(x.dropna().unique())[:3]),
    均价=("折后价", lambda x: round(x.dropna().mean(), 2))
).reset_index()

# -------------------------- 5. 输出结果 --------------------------
print("="*80)
print("Python图书分类归总结果")
print("="*80)
for _, row in classify_summary.iterrows():
    print(f"\n【{row['所属分类']}】")
    print(f"数量：{row['图书数量']}本 | 均价：{row['均价']}元 | 代表作者：{row['代表作者']}")
    print(f"图书列表：{row['图书列表']}")

# -------------------------- 6. 保存结果为CSV（核心修改处） --------------------------
# 保存分类后明细数据（用utf-8-sig编码，解决Excel打开CSV中文乱码问题）
detail_csv = "python_books_classified.csv"
df.to_csv(detail_csv, index=False, encoding="utf-8-sig")

# 保存分类汇总数据
summary_csv = "python_books_classify_summary.csv"
classify_summary.to_csv(summary_csv, index=False, encoding="utf-8-sig")

print("\n" + "="*80)
print(f"分类结果已保存为：")
print(f"1. 明细数据：{detail_csv}")
print(f"2. 汇总数据：{summary_csv}")

Python图书分类归总结果

【Python基础入门】
数量：26本 | 均价：83.7元 | 代表作者：['[美]埃里克・马瑟斯（EricMatthes）', '马国俊', '明日科技编著']
图书列表：['Python编程从入门到实践 第3版', 'Python网络爬虫与数据分析从入门到实践', 'Python完全自学教程', 'Python编程:从入门到实践(第3版)', '深度学习入门 基于Python的理论与实现', 'Python区块链量化交易', '笨办法学Python 原书第5版', '你好!Python', '小学生Python创意编程 视频教学版', '奇思妙想：Python青少年趣味编程101例（视频教学版）', 'Python从入门到精通 第3版', 'Python网络爬虫入门到实战', 'Python编程：从入门到实践', 'Python编程三剑客第3版：Python编程从入门到实践第3版+快速上手第2版+极客项目编程第2版（当当套装共3册）', '智能优化算法改进：从入门到MATLAB、Python编程实践', 'Python项目实战从入门到精通', '深度学习进阶 自然语言处理', 'Python AI游戏编程入门――基于Pygame和PyTorch', 'scikit-learn机器学习超入门', 'Python从入门到全栈开发', '机器学习 全彩图解 + 微课 + Python编程 鸢尾花数学大系：从加减乘除到机器学习', 'Python从入门到精通(第3版)', 'Python数据可视化从入门到项目实践（超值版）', 'Python高效办公――玩转Excel数据分析', '深度学习入门4：强化学习', '语音与音乐信号处理轻松入门（基于Python与PyTorch）']

【其他/跨领域融合】
数量：2本 | 均价：63.1元 | 代表作者：['朱宁', '郭屹']
图书列表：['Python自动化办公很简单', '认识编程――以Python语言讲透编程的本质']

【办公自动化】
数量：2本 | 均价：43.64元 | 代表作者：['王国平', '杨永刚']
图书列表：['Python+Office:轻松实现Python办公自动化', 'Python+ChatGPT办公自动化实战']

【数据分析与可视化】
数量：3本 | 